# Vitals 360 AI - audio model (normal / wheeze / crackle / cough-abnormal)

Small CNN on **log-mel** features that match `src/inference/mel.ts` exactly (16 kHz, n_fft 1024, hop 320, 64 Slaney mels, `log(1e-10 + power)`, 501 frames = 10 s). Trains on ICBHI 2017 (stethoscope, cycle labels) + COUGHVID expert-labelled phone coughs, exports ONNX, checks parity, writes dashboard metrics and a mel parity fixture for the TypeScript test.

**Before you run** (Kaggle): Accelerator GPU (T4/P100; a CPU run also works for this small model but is slower), Internet **On**. Add input `vbookshelf/respiratory-sound-database` (ICBHI). COUGHVID is downloaded from Zenodo by cell 5 (951 MB). First run: `SMOKE = True`.

HONEST WARNING: ICBHI is chest-stethoscope audio, the app records phone-microphone audio. Wheeze/crackle labels that were measured on stethoscope data may not transfer to a phone at the mouth/chest. This is a screening *prototype*; see `docs/data-report.md` section 5. Runtimes are estimates, not measured.

In [ ]:
!pip install -q onnx onnxruntime onnxscript tensorboard soundfile librosa

In [ ]:
import os, sys
os.makedirs("common", exist_ok=True)
sys.path.insert(0, os.getcwd())

In [ ]:
%%writefile common/__init__.py
"""Shared helpers for the Vitals 360 AI Kaggle notebooks (embedded into the .ipynb files by build_notebooks.py)."""

In [ ]:
%%writefile common/seed.py
import os
import random

import numpy as np
import torch


def set_seed(seed: int = 42) -> None:
    """Seed python, numpy and torch (CPU + CUDA). cudnn.benchmark stays on for speed; results are
    reproducible up to GPU non-determinism."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def worker_init(worker_id: int) -> None:
    s = torch.initial_seed() % 2**32
    np.random.seed(s)
    random.seed(s)

In [ ]:
%%writefile common/ckpt.py
import os
from pathlib import Path

import torch


def save_atomic(obj, path) -> None:
    """Write to a temp file then rename, so a kill mid-write never corrupts the checkpoint."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def load_if_exists(path, map_location="cpu"):
    path = Path(path)
    if not path.exists():
        return None
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except Exception as e:  # truncated file after a hard kill
        print(f"WARNING: could not read {path}: {e}")
        return None

In [ ]:
%%writefile common/calib.py
import numpy as np
import torch


def softmax(z: np.ndarray, T: float = 1.0) -> np.ndarray:
    z = z / T
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)


def nll(logits: np.ndarray, y: np.ndarray, T: float = 1.0) -> float:
    p = softmax(logits, T)
    return float(-np.log(np.clip(p[np.arange(len(y)), y], 1e-12, 1)).mean())


def fit_temperature(logits: np.ndarray, y: np.ndarray) -> float:
    """Temperature scaling (Guo et al. 2017): minimise NLL over a single scalar T on the VALIDATION set."""
    z = torch.tensor(logits, dtype=torch.float64)
    t = torch.tensor(y, dtype=torch.long)
    log_t = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=200)

    def closure():
        opt.zero_grad()
        loss = torch.nn.functional.cross_entropy(z / log_t.exp(), t)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp().clamp(0.05, 20.0).item())


def ece(probs: np.ndarray, y: np.ndarray, bins: int = 10) -> float:
    conf = probs.max(1)
    pred = probs.argmax(1)
    edges = np.linspace(0, 1, bins + 1)
    out = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            out += m.mean() * abs((pred[m] == y[m]).mean() - conf[m].mean())
    return float(out)


def coverage_table(probs: np.ndarray, y: np.ndarray, thresholds=None):
    """For each confidence threshold: fraction of cases answered, accuracy on those answered."""
    thresholds = thresholds if thresholds is not None else np.arange(0.3, 0.96, 0.05)
    conf, pred = probs.max(1), probs.argmax(1)
    rows = []
    for th in thresholds:
        m = conf >= th
        rows.append({"threshold": round(float(th), 2), "coverage": float(m.mean()),
                     "accuracy": float((pred[m] == y[m]).mean()) if m.any() else None})
    return rows


def pick_threshold(rows, min_coverage: float = 0.6, target_acc: float = 0.85) -> float:
    """Smallest threshold whose answered-accuracy >= target_acc while coverage >= min_coverage.
    Falls back to the best-accuracy row that still meets min_coverage. Fit on VALIDATION only."""
    ok = [r for r in rows if r["accuracy"] is not None and r["coverage"] >= min_coverage]
    good = [r for r in ok if r["accuracy"] >= target_acc]
    if good:
        return good[0]["threshold"]
    return max(ok, key=lambda r: r["accuracy"])["threshold"] if ok else 0.5

In [ ]:
%%writefile common/metrics.py
"""Metrics + the dashboard JSON (schema: eval/metrics/README.md, schemaVersion 1)."""
import datetime as _dt

import numpy as np


def confusion(y: np.ndarray, p: np.ndarray, k: int) -> np.ndarray:
    cm = np.zeros((k, k), dtype=int)
    for a, b in zip(y, p):
        cm[int(a), int(b)] += 1
    return cm


def per_class(cm: np.ndarray):
    """One-vs-rest sensitivity (recall), specificity, precision, f1 per class."""
    n = cm.sum()
    out = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = n - tp - fn - fp
        sens = tp / (tp + fn) if tp + fn else float("nan")
        spec = tn / (tn + fp) if tn + fp else float("nan")
        prec = tp / (tp + fp) if tp + fp else float("nan")
        f1 = 2 * prec * sens / (prec + sens) if (prec + sens) and not np.isnan(prec + sens) else float("nan")
        out.append({"sensitivity": sens, "specificity": spec, "precision": prec, "f1": f1, "support": int(tp + fn)})
    return out


def summary(cm: np.ndarray):
    pc = per_class(cm)
    mean = lambda key: float(np.nanmean([c[key] for c in pc]))  # noqa: E731
    return {"accuracy": float(np.trace(cm) / max(1, cm.sum())), "sensitivity": mean("sensitivity"),
            "specificity": mean("specificity"), "f1": mean("f1")}, pc


def _clean(x):
    return None if (isinstance(x, float) and np.isnan(x)) else round(float(x), 4)


def dashboard_entry(label, classes, y, pred, latency_ms, size_bytes, test_desc):
    cm = confusion(y, pred, len(classes))
    s, pc = summary(cm)
    return {
        "label": label, "classes": list(classes),
        "accuracy": _clean(s["accuracy"]), "sensitivity": _clean(s["sensitivity"]),
        "specificity": _clean(s["specificity"]), "f1": _clean(s["f1"]),
        "perClass": {c: {"sensitivity": _clean(pc[i]["sensitivity"]), "specificity": _clean(pc[i]["specificity"])}
                     for i, c in enumerate(classes)},
        "confusionMatrix": cm.tolist(), "latencyMs": round(float(latency_ms), 1),
        "sizeBytes": int(size_bytes), "testSetDescription": test_desc,
    }


def dashboard_file(model_id, entry, description):
    return {"schemaVersion": 1, "isSample": False,
            "generatedAt": _dt.datetime.now(_dt.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "description": description, "models": {model_id: entry}}


def subgroup_report(groups, y, pred, classes, min_n: int = 30):
    """Per-subgroup (e.g. Fitzpatrick type, data source) accuracy and per-class sensitivity.
    Groups with fewer than min_n samples are reported with a 'too_few' flag instead of numbers."""
    groups = np.asarray(groups)
    out = {}
    for g in sorted(set(groups.tolist())):
        m = groups == g
        n = int(m.sum())
        if n < min_n:
            out[str(g)] = {"n": n, "too_few": True}
            continue
        cm = confusion(y[m], pred[m], len(classes))
        pc = per_class(cm)
        out[str(g)] = {"n": n, "accuracy": _clean(np.trace(cm) / n),
                       "sensitivity": {c: _clean(pc[i]["sensitivity"]) for i, c in enumerate(classes)},
                       "support": {c: pc[i]["support"] for i, c in enumerate(classes)}}
    return out

In [ ]:
%%writefile common/train.py
"""Generic resumable training loop (AMP, checkpoint every epoch, TensorBoard, early stopping)."""
import time
from pathlib import Path

import numpy as np
import torch

from .ckpt import load_if_exists, save_atomic
from .metrics import confusion, summary

DEV = "cuda" if torch.cuda.is_available() else "cpu"


@torch.no_grad()
def predict(model, dl, prep):
    """Return (logits [N,K], labels [N]) in dataloader order (use shuffle=False)."""
    model.eval()
    zs, ys = [], []
    for x, y in dl:
        x = prep(x.to(DEV, non_blocking=True), train=False)
        with torch.autocast(device_type=DEV, dtype=torch.float16, enabled=DEV == "cuda"):
            z = model(x)
        zs.append(z.float().cpu().numpy())
        ys.append(y.numpy())
    return np.concatenate(zs), np.concatenate(ys)


def fit(model, train_dl, val_dl, crit, opt, sched, epochs, out_dir, prep, n_classes, writer=None, patience=6):
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    model.to(DEV)
    scaler = torch.amp.GradScaler("cuda", enabled=DEV == "cuda")
    start, best, bad = 0, -1.0, 0
    ck = load_if_exists(out / "last.pt")
    if ck:  # RESUME after a disconnect / restart
        model.load_state_dict(ck["model"])
        opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"])
        start, best, bad = ck["epoch"] + 1, ck["best"], ck["bad"]
        print(f"Resumed from epoch {ck['epoch']} (best val macro-F1 {best:.4f})")
    for ep in range(start, epochs):
        t0 = time.time()
        model.train()
        tot, n = 0.0, 0
        for x, y in train_dl:
            x = prep(x.to(DEV, non_blocking=True), train=True)
            y = y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEV, dtype=torch.float16, enabled=DEV == "cuda"):
                loss = crit(model(x), y)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            scaler.step(opt)
            scaler.update()
            tot += loss.item() * len(y)
            n += len(y)
        sched.step()
        z, yv = predict(model, val_dl, prep)
        s, _ = summary(confusion(yv, z.argmax(1), n_classes))
        f1 = s["f1"]
        if writer:
            writer.add_scalar("train/loss", tot / n, ep)
            writer.add_scalar("val/macro_f1", f1, ep)
            writer.add_scalar("val/acc", s["accuracy"], ep)
            writer.add_scalar("lr", opt.param_groups[0]["lr"], ep)
        improved = f1 > best
        if improved:
            best, bad = f1, 0
            save_atomic(model.state_dict(), out / "best.pt")
        else:
            bad += 1
        save_atomic({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                     "scaler": scaler.state_dict(), "epoch": ep, "best": best, "bad": bad}, out / "last.pt")
        print(f"epoch {ep + 1}/{epochs} loss {tot / n:.4f} val macroF1 {f1:.4f} acc {s['accuracy']:.4f} "
              f"{'*' if improved else ''} ({time.time() - t0:.0f}s)")
        if bad >= patience:
            print("early stop")
            break
    model.load_state_dict(torch.load(out / "best.pt", map_location=DEV))
    return best

In [ ]:
%%writefile common/export.py
"""ONNX export, int8 (QDQ static) / fp16 conversion, parity check, size + latency report, manifest patch."""
import hashlib
import json
import os
import time
from pathlib import Path

import numpy as np
import torch


def export_onnx(model, example: torch.Tensor, path, opset: int = 17) -> None:
    """Contract with src/inference/ort.ts: ONE input (float32), ONE output = raw logits [N, K]. No softmax,
    no temperature (the app applies manifest temperature itself)."""
    model = model.eval().cpu().float()
    kw = dict(input_names=["input"], output_names=["logits"], opset_version=opset,
              dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}})
    try:
        torch.onnx.export(model, example.cpu().float(), str(path), dynamo=False, **kw)
    except TypeError:  # older torch without the dynamo kwarg
        torch.onnx.export(model, example.cpu().float(), str(path), **kw)


def _session(path):
    import onnxruntime as ort
    return ort.InferenceSession(str(path), providers=["CPUExecutionProvider"])


def run_onnx(path, x: np.ndarray, bs: int = 32) -> np.ndarray:
    sess = _session(path)
    name = sess.get_inputs()[0].name
    return np.concatenate([sess.run(None, {name: x[i:i + bs].astype(np.float32)})[0] for i in range(0, len(x), bs)])


def quantize_int8(fp32_path, int8_path, calib: np.ndarray) -> None:
    """Static QDQ int8 quantisation using ~200 calibration inputs (no augmentation)."""
    from onnxruntime.quantization import CalibrationDataReader, QuantFormat, QuantType, quantize_static
    try:
        from onnxruntime.quantization.shape_inference import quant_pre_process
        pre = str(fp32_path).replace(".onnx", ".pre.onnx")
        quant_pre_process(str(fp32_path), pre, skip_symbolic_shape=True)
        src = pre
    except Exception as e:
        print("quant_pre_process skipped:", e)
        src = str(fp32_path)

    class Reader(CalibrationDataReader):
        def __init__(self):
            self.it = iter([{"input": c[None].astype(np.float32)} for c in calib])

        def get_next(self):
            return next(self.it, None)

    quantize_static(src, str(int8_path), Reader(), quant_format=QuantFormat.QDQ, per_channel=True,
                    weight_type=QuantType.QInt8, activation_type=QuantType.QUInt8)


def to_fp16(fp32_path, fp16_path) -> bool:
    try:
        import onnx
        from onnxconverter_common import float16
        m = onnx.load(str(fp32_path))
        onnx.save(float16.convert_float_to_float16(m, keep_io_types=True), str(fp16_path))
        return True
    except Exception as e:
        print("fp16 conversion failed:", e)
        return False


def sha256(path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def latency_ms(path, x1: np.ndarray, n: int = 50) -> float:
    """Median single-sample latency with ORT CPU (Kaggle CPU, NOT a phone: re-measure on a real phone)."""
    sess = _session(path)
    name = sess.get_inputs()[0].name
    for _ in range(5):
        sess.run(None, {name: x1})
    ts = []
    for _ in range(n):
        t = time.perf_counter()
        sess.run(None, {name: x1})
        ts.append((time.perf_counter() - t) * 1000)
    return float(np.median(ts))


def parity_report(torch_logits: np.ndarray, onnx_logits: np.ndarray):
    d = np.abs(torch_logits - onnx_logits)
    return {"max_abs_diff": float(d.max()), "mean_abs_diff": float(d.mean()),
            "top1_agreement": float((torch_logits.argmax(1) == onnx_logits.argmax(1)).mean())}


def manifest_patch(model_id, version, file_path, labels, temperature, threshold, extra_onnx=None):
    """The fields to paste into public/models/manifest.json for this model (tiers stay as already in the manifest)."""
    p = Path(file_path)
    patch = {"id": model_id, "version": version, "url": f"/models/{p.name}", "sha256": sha256(p),
             "bytes": os.path.getsize(p), "labels": labels, "threshold": round(threshold, 3),
             "temperature": round(temperature, 4)}
    if extra_onnx:
        patch["onnx"] = extra_onnx
    return patch


def dump(obj, path) -> None:
    with open(path, "w") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

In [ ]:
%%writefile common/mel.py
"""Log-mel front end that MUST match src/inference/mel.ts (MEL_DEFAULTS + logMel):
16 kHz, n_fft 1024, hop 320, periodic Hann (win 1024), centre=True with reflect padding, power=2,
64 mel bands 0..8000 Hz, Slaney scale + Slaney area norm, out = log(1e-10 + mel_power),
layout [frames][mels]; a 10 s clip (160000 samples) gives 1 + 160000//320 = 501 frames (= ort.ts default T)."""
import json

import librosa
import numpy as np
import torch

SR, N_FFT, HOP, N_MELS, EPS = 16000, 1024, 320, 64, 1e-10
FRAMES = 501
CLIP = (FRAMES - 1) * HOP  # 160000 samples = 10 s


class LogMel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("window", torch.hann_window(N_FFT, periodic=True))
        fb = librosa.filters.mel(sr=SR, n_fft=N_FFT, n_mels=N_MELS, fmin=0.0, fmax=SR / 2, htk=False, norm="slaney")
        self.register_buffer("fb", torch.from_numpy(fb.astype(np.float32)))

    @torch.no_grad()
    def forward(self, wav: torch.Tensor) -> torch.Tensor:
        """wav [B, L] float32 at 16 kHz -> [B, 1, frames, 64]."""
        spec = torch.stft(wav.float(), N_FFT, hop_length=HOP, win_length=N_FFT, window=self.window,
                          center=True, pad_mode="reflect", return_complex=True)
        power = spec.real ** 2 + spec.imag ** 2
        mel = torch.matmul(self.fb, power)  # [B, 64, T]
        return torch.log(EPS + mel).transpose(1, 2).unsqueeze(1)


def write_fixture(path, n_samples: int = 16000) -> dict:
    """Deterministic test signal + expected log-mel, for a vitest parity test against mel.ts (tolerance ~5e-3)."""
    rng = np.random.default_rng(0)
    t = np.arange(n_samples) / SR
    x = 0.3 * np.sin(2 * np.pi * 440 * t) + 0.1 * np.sin(2 * np.pi * (1500 + 800 * t) * t) + 0.01 * rng.standard_normal(n_samples)
    x = np.round(x, 6).astype(np.float32)
    m = LogMel()(torch.from_numpy(x)[None])[0, 0].numpy()
    fx = {"sampleRate": SR, "nFrames": int(m.shape[0]), "nMels": int(m.shape[1]),
          "samples": x.tolist(), "expected": np.round(m.reshape(-1), 5).tolist()}
    with open(path, "w") as f:
        json.dump(fx, f)
    return fx

## 1. Config

In [ ]:
import json, random, time, glob, shutil, zipfile, re, subprocess
from collections import Counter
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, librosa
from common.seed import set_seed, worker_init
from common.mel import LogMel, SR, CLIP, FRAMES, N_MELS, write_fixture

SMOKE = False
SEED = 42
EPOCHS = 2 if SMOKE else 40
BATCH = 64
CLASSES = ["normal", "wheeze", "crackle", "cough-abnormal"]   # MUST equal public/models/manifest.json audio labels
VERSION = "audio-v1-" + time.strftime("%Y%m%d")
WORK = Path("/kaggle/working"); OUT = WORK / "export"; CKPT = WORK / "ckpt_audio"; DATA = WORK / "data"
for d in (OUT, CKPT, DATA): d.mkdir(parents=True, exist_ok=True)
set_seed(SEED)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| frames", FRAMES, "| clip samples", CLIP)

## 2. ICBHI 2017 -> cycles

Each `.txt` row = `start end crackle wheeze`. Label: (0,0) normal, (1,0) crackle, (0,1) wheeze. **Cycles with both crackle and wheeze (~7%) are dropped** because the app has no "both" class. Group = patient id (first token of the file name).

In [ ]:
clips, meta = [], []   # clips: float32 mono 16 kHz arrays; meta: dict(y, group, source)
txts = [p for p in glob.glob("/kaggle/input/**/*.txt", recursive=True) if re.match(r"^\d{3}_\d+[a-z]\d_", os.path.basename(p))]
if not txts: raise SystemExit("ICBHI not found: + Add Input -> vbookshelf/respiratory-sound-database")
if SMOKE: txts = txts[:60]
for t in txts:
    wav = t[:-4] + ".wav"
    if not os.path.exists(wav): continue
    x, _ = librosa.load(wav, sr=SR, mono=True)
    pid = os.path.basename(t).split("_")[0]
    for line in open(t):
        s, e, c, w = line.split()[:4]; c, w = int(c), int(w)
        if c and w: continue
        seg = x[int(float(s) * SR): int(float(e) * SR)]
        if len(seg) < int(0.25 * SR): continue
        clips.append(seg.astype(np.float32)); meta.append(dict(y=1 if w else 2 if c else 0, group="icbhi_" + pid, source="ICBHI"))
print("ICBHI cycles:", len(clips), Counter(m["y"] for m in meta))

## 3. COUGHVID (phone microphones, crowdsourced) with expert labels

Downloaded from Zenodo record 4048312 (CC BY 4.0, 951 MB). Only recordings with at least one pulmonologist label are used. Mapping: majority expert diagnosis `healthy_cough` -> normal; `lower_infection` or `obstructive_disease` -> cough-abnormal; >=2 experts flag wheezing -> wheeze; everything else (`upper_infection`, `COVID-19`, unclear) is skipped. Column names are asserted at runtime (they were not verified when this was written).

In [ ]:
z = DATA / "coughvid.zip"
if not (DATA / "public_dataset").exists():
    if not z.exists() or z.stat().st_size < 9e8:
        subprocess.run(["curl", "-L", "-C", "-", "-o", str(z), "https://zenodo.org/records/4048312/files/public_dataset.zip?download=1"], check=True)
    zipfile.ZipFile(z).extractall(DATA)
mcsv = next(iter(glob.glob(str(DATA / "**/metadata_compiled.csv"), recursive=True)))
cv = pd.read_csv(mcsv); print(list(cv.columns))
dcols = [c for c in cv.columns if re.fullmatch(r"diagnosis_\d", c)]; wcols = [c for c in cv.columns if re.fullmatch(r"wheezing_\d", c)]
assert dcols, "expected diagnosis_1..4 expert columns; check the printed column list and adapt"
def cv_label(r):
    d = [r[c] for c in dcols if isinstance(r[c], str)]
    if not d: return None
    if sum(bool(r[c]) is True and r[c] == r[c] for c in wcols) >= 2: return 1
    top, n = Counter(d).most_common(1)[0]
    if n * 2 <= len(d): return None
    return {"healthy_cough": 0, "lower_infection": 3, "obstructive_disease": 3}.get(top)
n0 = len(clips); base = Path(mcsv).parent
if "cough_detected" in cv: cv = cv[cv.cough_detected.fillna(0) >= 0.8]
if SMOKE: cv = cv.head(300)
for r in cv.to_dict("records"):
    y = cv_label(r)
    if y is None: continue
    f = next((base / (r["uuid"] + e) for e in (".webm", ".ogg", ".wav") if (base / (r["uuid"] + e)).exists()), None)
    if f is None: continue
    try: x, _ = librosa.load(str(f), sr=SR, mono=True)
    except Exception: continue
    if len(x) < int(0.25 * SR): continue
    clips.append(x[:CLIP].astype(np.float32)); meta.append(dict(y=y, group="cv_" + r["uuid"], source="COUGHVID"))
print("COUGHVID clips used:", len(clips) - n0)
md = pd.DataFrame(meta); print(md.groupby(["source", "y"]).size().unstack(fill_value=0).rename(columns=dict(enumerate(CLASSES))))

## 4. Group-level split (patient for ICBHI, recording for COUGHVID: it has no user id)

~80/10/10. COUGHVID may contain several recordings of one person in different splits (known, small leak).

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
k = 10 if not SMOKE else 5
folds = list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=SEED).split(md, md.y, md.group))
sp = np.array(["train"] * len(md), dtype=object); sp[folds[0][1]] = "test"; sp[folds[1][1]] = "val"; md["split"] = sp
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    assert not set(md[md.split == a].group) & set(md[md.split == b].group), "group leak"
md.to_csv(WORK / "audio_splits.csv", index=False)
print(md.groupby(["split", "y"]).size().unstack(fill_value=0).rename(columns=dict(enumerate(CLASSES))))

## 5. 10-second canvases + phone-like augmentation

The app feeds a continuous 10 s recording (501 frames). Training samples are single cycles/coughs, so each is placed on a 10 s canvas: random offset, and with p=0.5 (always at eval) tiled with short gaps to mimic repeated breaths/coughs. Augmentation: gain, noise floor, random band-limit (phone mic), then SpecAugment on the mel. Gain does not matter at inference because the model subtracts the per-clip mean of the log-mel.

In [ ]:
from scipy.signal import butter, sosfilt
def canvas(clip, rng, train):
    L = CLIP; clip = clip[:L]
    if len(clip) < L // 2 and (not train or rng.random() < 0.5):
        gap = np.zeros(int((rng.uniform(0.1, 0.6) if train else 0.3) * SR), np.float32)
        clip = np.tile(np.concatenate([clip, gap]), L // (len(clip) + len(gap)) + 2)[:L]
    off = int(rng.integers(0, L - len(clip) + 1)) if train else 0
    out = np.zeros(L, np.float32); out[off: off + len(clip)] = clip
    peak = np.abs(out).max() + 1e-8
    out = out / peak * (10 ** rng.uniform(-1.6, -0.1) if train else 0.5)
    if train and rng.random() < 0.5:
        lo, hi = rng.uniform(60, 300), rng.uniform(3500, 7500)
        out = sosfilt(butter(4, [lo, hi], btype="band", fs=SR, output="sos"), out).astype(np.float32)
    out += rng.standard_normal(L).astype(np.float32) * (rng.uniform(5e-5, 2e-3) if train else 5e-4)
    return out

class AudioDS(torch.utils.data.Dataset):
    def __init__(self, idx, train): self.idx, self.train = np.asarray(idx), train
    def __len__(self): return len(self.idx)
    def __getitem__(self, i):
        j = self.idx[i]
        rng = np.random.default_rng() if self.train else np.random.default_rng(int(j))
        return torch.from_numpy(canvas(clips[j], rng, self.train)), int(Y[j])
Y = md.y.values
LOGMEL = LogMel().to(DEV)
def prep(x, train):
    m = LOGMEL(x)                                # [B,1,501,64]
    if train:                                    # SpecAugment: 2 freq masks, 2 time masks
        for _ in range(2):
            f = random.randint(0, 8); f0 = random.randint(0, N_MELS - f); m[:, :, :, f0:f0 + f] = m.mean()
            t = random.randint(0, 40); t0 = random.randint(0, FRAMES - t); m[:, :, t0:t0 + t, :] = m.mean()
    return m
def loader(name, train):
    idx = np.where(md.split.values == name)[0]
    return torch.utils.data.DataLoader(AudioDS(idx, train), batch_size=BATCH, shuffle=train, drop_last=train, num_workers=min(4, os.cpu_count()), worker_init_fn=worker_init)
train_dl, val_dl = loader("train", True), loader("val", False)

## 6. Model

Input = raw log-mel `[N,1,501,64]` exactly as `ort.ts` builds it. The model removes the per-clip mean and divides by a fixed std **inside the graph**, so the app does no normalisation.

In [ ]:
with torch.no_grad():
    sample = [prep(torch.from_numpy(canvas(clips[j], np.random.default_rng(j), True))[None].to(DEV), False) for j in np.where(md.split.values == "train")[0][:300]]
    STDV = float(torch.cat([s - s.mean() for s in sample]).std())
print("input std (after per-clip mean removal):", round(STDV, 3))

class Net(nn.Module):
    def __init__(self, k, std):
        super().__init__()
        self.register_buffer("std", torch.tensor(float(std)))
        def blk(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                                            nn.Conv2d(o, o, 3, padding=1, bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True), nn.MaxPool2d(2))
        self.f = nn.Sequential(blk(1, 16), blk(16, 32), blk(32, 64), blk(64, 128))
        self.drop = nn.Dropout(0.3); self.fc = nn.Linear(256, k)
    def forward(self, x):
        x = (x - x.mean(dim=(1, 2, 3), keepdim=True)) / self.std
        x = self.f(x)
        return self.fc(self.drop(torch.cat([x.mean(dim=(2, 3)), x.amax(dim=(2, 3))], 1)))

model = Net(len(CLASSES), STDV).to(DEV)
ytr = Y[md.split.values == "train"]; cnt = np.bincount(ytr, minlength=len(CLASSES))
cw = (len(ytr) / (len(CLASSES) * np.maximum(cnt, 1))) ** 0.5
print("train counts", cnt, "weights", cw.round(2), "| params", sum(p.numel() for p in model.parameters()))
crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32, device=DEV), label_smoothing=0.05)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

## 7. Train (resumable)

Estimate: ~15-40 s/epoch on a T4 (data loading bound), 40 epochs about 10-25 min. After a disconnect: Run All again, the loop resumes from `/kaggle/working/ckpt_audio/last.pt`. TensorBoard files: `/kaggle/working/runs/audio`.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from common.train import fit, predict
writer = SummaryWriter(str(WORK / "runs" / "audio"))
best = fit(model, train_dl, val_dl, crit, opt, sched, EPOCHS, CKPT, prep, len(CLASSES), writer=writer, patience=10)
writer.close(); print("best val macro-F1:", round(best, 4))

## 8. Calibration on validation (temperature + abstain threshold)

In [ ]:
from common.calib import softmax, fit_temperature, ece, nll, coverage_table, pick_threshold
zv, yv = predict(model, val_dl, prep)
T = fit_temperature(zv, yv)
rows = coverage_table(softmax(zv, T), yv, thresholds=np.arange(0.3, 0.9, 0.05))
THRESH = pick_threshold(rows, min_coverage=0.6, target_acc=0.8)
cal = {"temperature": T, "threshold": THRESH, "val_nll_before": nll(zv, yv), "val_nll_after": nll(zv, yv, T),
       "val_ece_before": ece(softmax(zv), yv), "val_ece_after": ece(softmax(zv, T), yv), "coverage_table_val": rows}
print({k: v for k, v in cal.items() if k != "coverage_table_val"})

## 9. Export ONNX + parity (mel inputs from the test split)

In [ ]:
from common.export import *
from common.metrics import confusion, summary, dashboard_entry, dashboard_file, subgroup_report
model = model.cpu().float().eval()
FP32 = OUT / "audio-fp32.onnx"
export_onnx(model, torch.zeros(1, 1, FRAMES, N_MELS), FP32)
te_idx = np.where(md.split.values == "test")[0]
tdl = loader("test", False); lm = LogMel()
Xte = np.concatenate([lm(w).numpy() for w, _ in tdl]).astype(np.float32); yte = np.concatenate([y.numpy() for _, y in tdl])
with torch.no_grad(): zt_torch = np.concatenate([model(torch.from_numpy(Xte[i:i + 64])).numpy() for i in range(0, len(Xte), 64)])
zt_fp32 = run_onnx(FP32, Xte); par = parity_report(zt_torch, zt_fp32); print("fp32 parity", par)
assert par["max_abs_diff"] < 1e-3, "ONNX export does not match PyTorch"

## 10. Optional int8, choose artifact

The model is ~1 MB in fp32, far under the 15 MB budget, so fp32 is shipped unless int8 is both smaller and equal in quality (top-1 agreement >= 0.97, macro-F1 drop <= 0.02) and fp32 is over 4 MB.

In [ ]:
f1_of = lambda z: summary(confusion(yte, z.argmax(1), len(CLASSES)))[0]["f1"]
cands = {"fp32": (FP32, zt_fp32, par)}
INT8 = OUT / "audio-int8.onnx"
try:
    quantize_int8(FP32, INT8, Xte[:min(150, len(Xte))]); z8 = run_onnx(INT8, Xte); cands["int8"] = (INT8, z8, parity_report(zt_torch, z8))
except Exception as e: print("int8 skipped:", e)
for n, (p, z, pr) in cands.items(): print(n, os.path.getsize(p), pr["top1_agreement"], round(f1_of(z), 4))
choice = "fp32"
if "int8" in cands and os.path.getsize(FP32) > 4_000_000 and cands["int8"][2]["top1_agreement"] >= 0.97 and f1_of(cands["int8"][1]) >= f1_of(zt_fp32) - 0.02: choice = "int8"
SHIP = OUT / "audio-v1.onnx"; shutil.copy(cands[choice][0], SHIP); z_ship = cands[choice][1]
print("shipping", choice, os.path.getsize(SHIP), "bytes")

## 11. Metrics, report, manifest patch, mel fixture

In [ ]:
lat = latency_ms(SHIP, Xte[:1])
te = md.iloc[te_idx].reset_index(drop=True); pred = z_ship.argmax(1)
desc = (f"Held-out group-level split (seed {SEED}), n={len(te)} 10 s canvases built from single ICBHI cycles / COUGHVID coughs; "
        + ", ".join(f"{k} {v}" for k, v in te.source.value_counts().items()) + ". Cycles with both crackle+wheeze excluded. Latency = ONNX Runtime CPU on Kaggle, not a phone.")
entry = dashboard_entry("Audio", CLASSES, yte, pred, lat, os.path.getsize(SHIP), desc)
dump(dashboard_file("audio", entry, f"Real measured run {VERSION}. Stethoscope + phone-cough data mixed; see docs/data-report.md."), OUT / "metrics_audio.json")
P = softmax(z_ship, T); ans = P.max(1) >= THRESH
report = {"version": VERSION, "choice": choice, "seed": SEED, "calibration": cal, "test_ece_after": ece(P, yte),
          "abstain": {"coverage": float(ans.mean()), "accuracy_answered": float((pred[ans] == yte[ans]).mean()) if ans.any() else None},
          "parity": {k: v[2] for k, v in cands.items()}, "by_source": subgroup_report(te.source.values, yte, pred, CLASSES),
          "caveat": "ICBHI = stethoscope; COUGHVID = phone. Class and recording device are partly confounded; trust by_source, not the pooled number."}
dump(report, OUT / "report_audio.json")
dump(manifest_patch("audio", VERSION, SHIP, CLASSES, T, THRESH, extra_onnx={"input": "input", "output": "logits", "frames": FRAMES}), OUT / "manifest_patch_audio.json")
fx = write_fixture(OUT / "mel_fixture.json"); print("fixture frames", fx["nFrames"], "mels", fx["nMels"])
with zipfile.ZipFile(WORK / "audio_bundle.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ["audio-v1.onnx", "metrics_audio.json", "report_audio.json", "manifest_patch_audio.json", "mel_fixture.json"]: zf.write(OUT / f, f)
print(json.dumps(entry, indent=1)[:1500])

## 12. Next steps

Download `audio_bundle.zip`. Copy `audio-v1.onnx` to `public/models/`, merge `manifest_patch_audio.json` into the audio entry of `public/models/manifest.json` (keep `tiers`), copy `metrics_audio.json` to `eval/metrics/`, and add `mel_fixture.json` as a vitest fixture for `src/inference/mel.ts` (compare `logMel(samples, 16000)` with `expected`, atol ~5e-3). Details: `docs/gpu-training-playbook.md` section 7.